## Data Preprocessing and Feature Engineering

After exploring the dataset, we prepared the data for machine learning models.

We checked for missing values, duplicate records, and outliers. Then, we split the data into training and testing sets and converted the categorical columns into numerical values.

We also applied scaling to selected numerical features and created a new feature, `total_extra_services`, to count the extra services requested by each customer.

The processed data was then saved for the model training stage.

In [32]:
import pandas as pd

url = "https://raw.githubusercontent.com/Ghaida-Alyousef/SW485-Project-Group5/main/Dataset/customer_booking.csv"

df = pd.read_csv(url, encoding="latin1")

print(df.shape)
df.head()

(50000, 14)


,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,New Zealand,1,0,0,5.52,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,New Zealand,0,0,0,5.52,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,India,1,1,0,5.52,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,New Zealand,0,0,1,5.52,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,India,1,0,1,5.52,0


In [33]:

# Check missing values and duplicate records

print("Missing values:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nFully duplicated rows:", df.duplicated().sum())

print("\nRows with duplicated input features:",
      df.duplicated(subset=df.columns.drop("booking_complete")).sum())


Missing values:
num_passengers           0
sales_channel            0
trip_type                0
purchase_lead            0
length_of_stay           0
flight_hour              0
flight_day               0
route                    0
booking_origin           0
wants_extra_baggage      0
wants_preferred_seat     0
wants_in_flight_meals    0
flight_duration          0
booking_complete         0
dtype: int64

Total missing values: 0

Fully duplicated rows: 719

Rows with duplicated input features: 727


### Missing Values and Duplicate Records

The dataset has no missing values, so no missing-value treatment was needed.

We found 719 fully duplicated rows and 727 rows with duplicated input features. Since there is no unique booking ID, we cannot confirm that these are duplicate bookings rather than different bookings with the same information.

We kept these records and used a group-aware train/test split to make sure identical input feature patterns do not appear in both sets.

In [34]:

# Inspect duplicate records and their target distribution

duplicate_rows = df[df.duplicated(keep=False)]

print("Total rows belonging to duplicate groups:",
      len(duplicate_rows))

print("\nBooking completion among duplicated rows:")
print(duplicate_rows["booking_complete"].value_counts())

print("\nSample duplicated records:")
display(duplicate_rows.head(10))


Total rows belonging to duplicate groups: 1265

Booking completion among duplicated rows:
booking_complete
0    1107
1     158
Name: count, dtype: int64

Sample duplicated records:


,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
115,1,Internet,RoundTrip,65,278,2,Thu,AKLKUL,Malaysia,0,0,0,8.83,0
117,1,Internet,RoundTrip,263,58,4,Sun,AKLKUL,Malaysia,0,1,0,8.83,0
122,1,Internet,RoundTrip,42,17,14,Tue,AKLKUL,Malaysia,0,0,0,8.83,1
135,4,Internet,RoundTrip,366,17,16,Sun,AKLKUL,Malaysia,1,0,1,8.83,0
177,5,Internet,RoundTrip,206,17,13,Sun,AKLKUL,Malaysia,1,0,0,8.83,0
188,1,Mobile,RoundTrip,139,22,8,Sun,AKLKUL,New Zealand,1,0,1,8.83,0
195,1,Internet,RoundTrip,380,21,21,Thu,AKLKUL,New Zealand,1,1,0,8.83,0
196,5,Internet,RoundTrip,206,17,13,Sun,AKLKUL,Malaysia,1,0,0,8.83,0
274,1,Internet,RoundTrip,172,54,8,Tue,AKLKUL,Malaysia,1,1,1,8.83,0
316,5,Internet,RoundTrip,206,17,13,Sun,AKLKUL,Malaysia,1,0,0,8.83,0


### Duplicate Records Analysis

We found 719 fully duplicated rows. When including the first occurrence of each duplicated record, 1,265 rows belong to duplicate groups.

Since the dataset does not include a unique booking ID, we cannot confirm whether these records are actual duplicates or separate bookings with identical information.

We kept these records because they may represent different bookings with the same information. We also used a group-aware train/test split to prevent identical input feature patterns from appearing in both sets.

In [35]:

# Detect potential outliers using the IQR method

columns_to_check = [
    "purchase_lead",
    "length_of_stay",
    "num_passengers"
]

for col in columns_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]

    print(f"{col}:")
    print(f"Lower bound: {lower_bound:.2f}")
    print(f"Upper bound: {upper_bound:.2f}")
    print(f"Potential outliers: {len(outliers)}")
    print()


purchase_lead:
Lower bound: -120.00
Upper bound: 256.00
Potential outliers: 3456

length_of_stay:
Lower bound: -29.50
Upper bound: 62.50
Potential outliers: 3807

num_passengers:
Lower bound: -0.50
Upper bound: 3.50
Potential outliers: 2903



### Outlier Analysis

We used the IQR method to check for potential outliers in `purchase_lead`, `length_of_stay`, and `num_passengers`.

The results showed 3,456, 3,807, and 2,903 potential outliers, respectively.

These values are not necessarily errors. For example, bookings with four or more passengers can be valid.

We decided to keep these records for now to avoid losing potentially useful information.

In [36]:

# Check categorical columns before encoding

categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head())
    print()


sales_channel: 2 unique values
sales_channel
Internet    44382
Mobile       5618
Name: count, dtype: int64

trip_type: 3 unique values
trip_type
RoundTrip     49497
OneWay          387
CircleTrip      116
Name: count, dtype: int64

flight_day: 7 unique values
flight_day
Mon    8102
Wed    7674
Tue    7673
Thu    7424
Fri    6761
Name: count, dtype: int64

route: 799 unique values
route
AKLKUL    2680
PENTPE     924
MELSGN     842
ICNSIN     801
DMKKIX     744
Name: count, dtype: int64

booking_origin: 104 unique values
booking_origin
Australia      17872
Malaysia        7174
South Korea     4559
Japan           3885
China           3387
Name: count, dtype: int64



### Categorical Feature Analysis

We checked the number of unique values in each categorical column before encoding.

The columns `sales_channel`, `trip_type`, and `flight_day` have relatively few categories, so One-Hot Encoding is suitable for them.

However, `route` has 799 unique values and `booking_origin` has 104. We plan to use Frequency Encoding for these columns to avoid creating too many new features.

### Initial Train/Test Split

We first used an 80/20 stratified train/test split to keep the class proportions similar.

After checking both sets, we found 191 identical feature patterns shared between training and testing. We then used GroupShuffleSplit to prevent this overlap.

The group-aware split is the final split used for preprocessing.

In [37]:

from sklearn.model_selection import train_test_split

# Separate input features and target
X = df.drop(columns=["booking_complete"])
y = df["booking_complete"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)


Training set: (40000, 13)
Testing set: (10000, 13)

Training target distribution:
booking_complete
0    85.045
1    14.955
Name: proportion, dtype: float64

Testing target distribution:
booking_complete
0    85.04
1    14.96
Name: proportion, dtype: float64


In [38]:

# Check for identical booking features in both sets

train_features = X_train.drop_duplicates()
test_features = X_test.drop_duplicates()

overlap = test_features.merge(
    train_features,
    how="inner",
    on=list(X_train.columns)
)

print("Unique feature patterns shared by train and test:", len(overlap))

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


Unique feature patterns shared by train and test: 191

Training rows: 40000
Testing rows: 10000


### Final Train/Test Split

We used GroupShuffleSplit to keep records with identical input features in the same set.

The final split contains 39,989 training records and 10,011 testing records. After checking the two sets, we found no shared feature patterns.

The booking completion rates were also similar in both sets, at around 15%.

In [39]:

from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# Use identical input features as a group
groups = pd.util.hash_pandas_object(
    df.drop(columns=["booking_complete"]),
    index=False
)

# Split groups into training and testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, df["booking_complete"], groups=groups)
)

X = df.drop(columns=["booking_complete"])
y = df["booking_complete"]

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Verify no identical feature patterns exist in both sets
overlap = X_test.drop_duplicates().merge(
    X_train.drop_duplicates(),
    how="inner",
    on=list(X.columns)
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("Shared feature patterns:", len(overlap))

print("\nTraining completion rate:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting completion rate:")
print(y_test.value_counts(normalize=True) * 100)


Training set: (39989, 13)
Testing set: (10011, 13)
Shared feature patterns: 0

Training completion rate:
booking_complete
0    85.068394
1    14.931606
Name: proportion, dtype: float64

Testing completion rate:
booking_complete
0    84.946559
1    15.053441
Name: proportion, dtype: float64


In [40]:

from sklearn.preprocessing import OneHotEncoder

low_cardinality_cols = [
    "sales_channel",
    "trip_type",
    "flight_day"
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Learn categories from training data only
encoder.fit(X_train[low_cardinality_cols])

# Transform both sets using the same encoder
train_encoded = encoder.transform(
    X_train[low_cardinality_cols]
)

test_encoded = encoder.transform(
    X_test[low_cardinality_cols]
)

print("Training encoded shape:", train_encoded.shape)
print("Testing encoded shape:", test_encoded.shape)

print("\nEncoded columns:")
print(encoder.get_feature_names_out())


Training encoded shape: (39989, 12)
Testing encoded shape: (10011, 12)

Encoded columns:
['sales_channel_Internet' 'sales_channel_Mobile' 'trip_type_CircleTrip'
 'trip_type_OneWay' 'trip_type_RoundTrip' 'flight_day_Fri'
 'flight_day_Mon' 'flight_day_Sat' 'flight_day_Sun' 'flight_day_Thu'
 'flight_day_Tue' 'flight_day_Wed']


### One-Hot Encoding

We apply One-Hot Encoding to low-cardinality categorical features: `sales_channel`, `trip_type`, and `flight_day`.

The encoder is fitted on the training data only and then applied to both training and testing data to prevent data leakage.

In [41]:

# Frequency Encoding for high-cardinality columns

high_cardinality_cols = [
    "route",
    "booking_origin"
]

X_train_freq = X_train.copy()
X_test_freq = X_test.copy()

for col in high_cardinality_cols:
    # Calculate frequencies using training data only
    frequency_map = X_train[col].value_counts(normalize=True)

    # Apply the same mapping to training and testing
    X_train_freq[col] = X_train[col].map(frequency_map)
    X_test_freq[col] = X_test[col].map(frequency_map).fillna(0)

print("Training shape:", X_train_freq.shape)
print("Testing shape:", X_test_freq.shape)

print("\nSample encoded values:")
display(X_train_freq[high_cardinality_cols].head())


Training shape: (39989, 13)
Testing shape: (10011, 13)

Sample encoded values:


,route,booking_origin
1,0.0004,0.021681
2,0.0004,0.025007
3,0.0004,0.021681
4,0.0004,0.025007
5,0.0004,0.021681


### Frequency Encoding

We used Frequency Encoding for `route` and `booking_origin` because they have many unique categories. This helps avoid creating a large number of columns.

The frequencies were calculated from the training data only and then applied to both training and testing sets. Categories not found in the training data were assigned a value of zero.

One limitation of this method is that different categories may have the same frequency value.

In [42]:

# Get names of One-Hot encoded columns
encoded_col_names = encoder.get_feature_names_out(
    low_cardinality_cols
)

# Convert One-Hot arrays into DataFrames
train_onehot_df = pd.DataFrame(
    train_encoded,
    columns=encoded_col_names,
    index=X_train.index
)

test_onehot_df = pd.DataFrame(
    test_encoded,
    columns=encoded_col_names,
    index=X_test.index
)

# Remove original low-cardinality text columns
X_train_ready = X_train_freq.drop(
    columns=low_cardinality_cols
)

X_test_ready = X_test_freq.drop(
    columns=low_cardinality_cols
)

# Combine all encoded features
X_train_ready = pd.concat(
    [X_train_ready, train_onehot_df],
    axis=1
)

X_test_ready = pd.concat(
    [X_test_ready, test_onehot_df],
    axis=1
)

print("Training shape:", X_train_ready.shape)
print("Testing shape:", X_test_ready.shape)

print("\nRemaining text columns:")
print(X_train_ready.select_dtypes(include="object").columns.tolist())

display(X_train_ready.head())


Training shape: (39989, 22)
Testing shape: (10011, 22)

Remaining text columns:
[]


,num_passengers,purchase_lead,length_of_stay,flight_hour,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,...,trip_type_CircleTrip,trip_type_OneWay,trip_type_RoundTrip,flight_day_Fri,flight_day_Mon,flight_day_Sat,flight_day_Sun,flight_day_Thu,flight_day_Tue,flight_day_Wed
1,1,112,20,3,0.0004,0.021681,0,0,0,5.52,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2,243,22,17,0.0004,0.025007,1,1,0,5.52,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,96,31,4,0.0004,0.021681,0,0,1,5.52,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,2,68,22,15,0.0004,0.025007,1,0,1,5.52,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,1,3,48,20,0.0004,0.021681,1,0,1,5.52,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [43]:

# Feature Engineering: total number of extra services

service_cols = [
    "wants_extra_baggage",
    "wants_preferred_seat",
    "wants_in_flight_meals"
]

X_train_ready["total_extra_services"] = (
    X_train_ready[service_cols].sum(axis=1)
)

X_test_ready["total_extra_services"] = (
    X_test_ready[service_cols].sum(axis=1)
)

print("Training shape:", X_train_ready.shape)
print("Testing shape:", X_test_ready.shape)

print("\nNew feature distribution in training:")
print(X_train_ready["total_extra_services"].value_counts().sort_index())

display(X_train_ready[
    service_cols + ["total_extra_services"]
].head())


Training shape: (39989, 23)
Testing shape: (10011, 23)

New feature distribution in training:
total_extra_services
0     8393
1    14487
2    10109
3     7000
Name: count, dtype: int64


,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,total_extra_services
1,0,0,0,0
2,1,1,0,2
3,0,0,1,1
4,1,0,1,2
5,1,0,1,2


### Feature Engineering: Total Extra Services

We created a new feature called `total_extra_services` by summing three binary variables: extra baggage, preferred seat, and in-flight meals.

The new feature ranges from 0 to 3 and represents the number of optional services requested by each customer.

This feature may help the models capture overall customer interest in additional flight services. Its usefulness will be evaluated during model training.

In [44]:

from sklearn.preprocessing import StandardScaler

# Numerical features to scale
columns_to_scale = [
    "num_passengers",
    "purchase_lead",
    "length_of_stay",
    "flight_hour",
    "flight_duration",
    "total_extra_services"
]

scaler = StandardScaler()

# Create separate copies
X_train_final = X_train_ready.copy()
X_test_final = X_test_ready.copy()

# Fit only on training data
X_train_final[columns_to_scale] = scaler.fit_transform(
    X_train_ready[columns_to_scale]
)

# Apply the same scaling to testing data
X_test_final[columns_to_scale] = scaler.transform(
    X_test_ready[columns_to_scale]
)

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

print("\nScaled training feature means:")
print(X_train_final[columns_to_scale].mean().round(3))

print("\nMissing values after preprocessing:")
print("Training:", X_train_final.isna().sum().sum())
print("Testing:", X_test_final.isna().sum().sum())


Final training shape: (39989, 23)
Final testing shape: (10011, 23)

Scaled training feature means:
num_passengers         -0.0
purchase_lead          -0.0
length_of_stay         -0.0
flight_hour             0.0
flight_duration         0.0
total_extra_services   -0.0
dtype: float64

Missing values after preprocessing:
Training: 0
Testing: 0


### Feature Scaling

We used StandardScaler to scale six numerical features: `num_passengers`, `purchase_lead`, `length_of_stay`, `flight_hour`, `flight_duration`, and `total_extra_services`.

The scaler was fitted on the training data only and then applied to the testing data. This helps prevent data leakage.

We left the binary and One-Hot encoded features unchanged. StandardScaler changes the scale of numerical values, but it does not remove outliers.

In [45]:

# Save preprocessed training and testing datasets

train_output = X_train_final.copy()
test_output = X_test_final.copy()

# Add target column without changing its values
train_output["booking_complete"] = y_train
test_output["booking_complete"] = y_test

# Final checks
print("Training shape:", train_output.shape)
print("Testing shape:", test_output.shape)

print("Training missing values:", train_output.isna().sum().sum())
print("Testing missing values:", test_output.isna().sum().sum())

print("Training target values:",
      sorted(train_output["booking_complete"].unique()))

print("Testing target values:",
      sorted(test_output["booking_complete"].unique()))

# Export files
train_output.to_csv("train_preprocessed.csv", index=False)
test_output.to_csv("test_preprocessed.csv", index=False)

print("\nFiles saved successfully!")


Training shape: (39989, 24)
Testing shape: (10011, 24)
Training missing values: 0
Testing missing values: 0
Training target values: [np.int64(0), np.int64(1)]
Testing target values: [np.int64(0), np.int64(1)]

Files saved successfully!


In [46]:
preprocessed_data = pd.concat(
    [train_output.assign(data_split="train"),
     test_output.assign(data_split="test")],
    axis=0
).sort_index()

preprocessed_data.to_csv("preprocessed_data.csv", index=False)

print("Shape:", preprocessed_data.shape)
print(preprocessed_data["data_split"].value_counts())

Shape: (50000, 25)
data_split
train    39989
test     10011
Name: count, dtype: int64


### Exporting Preprocessed Data

After encoding, feature engineering, and scaling, the final dataset contains 23 numerical input features and one target column, `booking_complete`.

We exported the training and testing sets separately as `train_preprocessed.csv` and `test_preprocessed.csv` to preserve the group-aware split and allow the team to use the same processed data.

All preprocessing statistics were learned from the training data only to reduce the risk of data leakage.

We also created `preprocessed_data.csv` by combining the training and testing sets. This file contains 50,000 records and includes a `data_split` column to identify which records belong to each set. The `data_split` column is only used to identify the split and should not be used as an input feature when training the models.